# Crop Type Classification from Satellite Data

**Goal:** predict the crop type (7 classes) from radar and optical satellite features.
**Data:** UCI *Crop mapping using fused optical-radar data set* (325,834 rows, CC BY 4.0).
**How to run:** set the runtime to **GPU**, then choose *Runtime → Run all*. Two switches in Section 1 control the run time.

## 1. Setup
Imports, random seeds, and settings. Expected output: library versions and the GPU list.

In [ ]:
import os, random, time, json, shutil, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, average_precision_score, precision_recall_curve,
                             classification_report, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

# ---- settings -------------------------------------------------------
RUN_RANDOM_FOREST = False   # True adds Random Forest (E2), about 15 extra minutes
N_TUNING_RUNS = 8           # number of boosting settings to try in Section 7
MIN_ROWS = 20_000           # project rule: at least 20,000 raw rows
DATA_DIR, MODEL_DIR, LOG_DIR = "data", "models", "logs"

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Load the data
Download the official zip from UCI (it is also a Drive deliverable) and read it into a table. Expected output: shape about (325834, 175) and a passed row check.

In [ ]:
ZIP_PATH = f"{DATA_DIR}/crop_mapping.zip"
URL = "https://archive.ics.uci.edu/static/public/525/crop+mapping+using+fused+optical+radar+data+set.zip"

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(ZIP_PATH):
    urllib.request.urlretrieve(URL, ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_DIR)

files = [f for f in os.listdir(DATA_DIR) if f.endswith((".txt", ".csv"))]
DATA_FILE = os.path.join(DATA_DIR, max(files, key=lambda f: os.path.getsize(os.path.join(DATA_DIR, f))))
df = pd.read_csv(DATA_FILE)

TARGET = "label"
assert TARGET in df.columns, "Label column not found"
feature_cols = [c for c in df.columns if c != TARGET]

print("Shape:", df.shape, "| features:", len(feature_cols))
assert len(df) >= MIN_ROWS, "Below the 20,000-row requirement"
print("Row-count requirement met.")

## 3. Explore the data
Check missing values, duplicates, and class balance. Expected output: no missing values or duplicates, and a strongly imbalanced class table (class 7 is only 0.35%).

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

counts = df[TARGET].value_counts().sort_index()
print(pd.DataFrame({"count": counts, "percent": (counts / len(df) * 100).round(2)}))

plt.figure(figsize=(8, 4))
sns.countplot(x=df[TARGET], order=sorted(df[TARGET].unique()))
plt.title("Crop class distribution"); plt.xlabel("Crop class"); plt.ylabel("Samples")
plt.tight_layout(); plt.show()

### Feature scales, skew, and redundancy
Feature ranges differ hugely, many features are skewed, and many are strongly correlated. This drives the preprocessing choices later. Expected output: three short summary lines.

In [ ]:
stats = df[feature_cols].agg(["min", "max"]).T
stats["range"] = stats["max"] - stats["min"]
print(f"Feature ranges: {stats['range'].min():.4f} to {stats['range'].max():.0f}")

skew = df[feature_cols].skew().abs()
print("Features with |skew| > 2:", int((skew > 2).sum()), "of", len(feature_cols))

corr = df[feature_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
print("Feature pairs with |corr| > 0.95:", int((upper > 0.95).sum().sum()))

## 4. Split the data
First a normal random split (stratified 70/15/15) as a reference. Expected output: split sizes and matching class percentages.

In [ ]:
X_all, y_all = df[feature_cols], df[TARGET]
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.30, stratify=y_all, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
print("Random split -> train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)

### Leakage check
Rows are pixels from satellite images. If the file is sorted and neighbouring rows are similar, a random split leaks information. Expected output: only a few label runs (the file is sorted by class) and high lag-1 autocorrelation inside each class (about 0.6 to 0.8) versus about 0 when shuffled.

In [ ]:
labels = df[TARGET].to_numpy()
runs = int((labels[1:] != labels[:-1]).sum()) + 1
print("Label runs in file order:", runs)

rng = np.random.default_rng(SEED)
sample_feats = rng.choice(feature_cols, size=20, replace=False)
lag1 = lambda x: np.corrcoef(x[:-1], x[1:])[0, 1]

rows = []
for c in sorted(df[TARGET].unique()):
    block = df.loc[df[TARGET] == c, sample_feats]
    rows.append({"class": c,
                 "lag1_real": np.nanmean([lag1(block[f].to_numpy()) for f in sample_feats]),
                 "lag1_shuffled": np.nanmean([lag1(rng.permutation(block[f].to_numpy())) for f in sample_feats])})
print(pd.DataFrame(rows).round(3).to_string(index=False))

### Block split (the stricter split used for the main experiments)
Inside each class, the first 70% of rows go to train, the next 15% to validation, and the last 15% to test. A 1% gap is dropped between blocks so neighbouring pixels do not straddle a cut. Expected output: split sizes and matching class percentages.

In [ ]:
tr_idx, va_idx, te_idx = [], [], []
for c in sorted(df[TARGET].unique()):
    idx = df.index[df[TARGET] == c].to_numpy()
    n = len(idx); buf = max(1, int(0.01 * n))
    a, b = int(0.70 * n), int(0.85 * n)
    tr_idx.extend(idx[: a - buf])
    va_idx.extend(idx[a + buf: b - buf])
    te_idx.extend(idx[b + buf:])

Xb_train, yb_train = df.loc[tr_idx, feature_cols], df.loc[tr_idx, TARGET]
Xb_val,   yb_val   = df.loc[va_idx, feature_cols], df.loc[va_idx, TARGET]
Xb_test,  yb_test  = df.loc[te_idx, feature_cols], df.loc[te_idx, TARGET]

print("Block split -> train:", Xb_train.shape, "val:", Xb_val.shape, "test:", Xb_test.shape)
print((yb_val.value_counts(normalize=True).sort_index() * 100).round(2).to_string())

## 5. Helper functions
One function trains a model and adds a row to the experiment table, so every run is recorded the same way. The main metric is **macro-F1** (it treats all classes equally, which matters because the classes are imbalanced). Expected output: none.

In [ ]:
def signed_log1p(x):
    """Log transform that keeps the sign; tames very skewed features."""
    return np.sign(x) * np.log1p(np.abs(x))

# Column layout from the UCI variable table: f1-f49 radar (5 Jul), f50-f98 radar (14 Jul),
# f99-f136 optical (5 Jul), f137-f174 optical (14 Jul); features pair by position.
RADAR_D1, RADAR_D2 = slice(0, 49), slice(49, 98)
OPT_D1, OPT_D2 = slice(98, 136), slice(136, 174)

def add_date_diffs(X):
    """Append (date 2 - date 1) for every radar and optical feature."""
    A = np.asarray(X, dtype=float)
    return np.hstack([A, A[:, RADAR_D2] - A[:, RADAR_D1], A[:, OPT_D2] - A[:, OPT_D1]])

SPLITS = {
    "random": dict(Xtr=X_train, ytr=y_train, Xva=X_val, yva=y_val,
                   desc="random stratified 70/15/15 (seed 42)"),
    "block":  dict(Xtr=Xb_train, ytr=yb_train, Xva=Xb_val, yva=yb_val,
                   desc="within-class blocks 70/15/15, 1% buffers"),
}

experiment_log = []

def log_row(exp_id, name, preprocessing, settings, split_desc,
            y_true, y_pred, proba, train_time, notes=""):
    row = {
        "exp_id": exp_id, "model": name, "preprocessing": preprocessing,
        "settings": settings, "split": split_desc,
        "val_accuracy": accuracy_score(y_true, y_pred),
        "val_macro_f1": f1_score(y_true, y_pred, average="macro"),
        "val_macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "val_macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "val_roc_auc_ovr": roc_auc_score(y_true, proba, multi_class="ovr", average="macro"),
        "train_time_s": round(train_time, 1), "notes": notes,
    }
    experiment_log.append(row)
    print(f"{exp_id} | {name}: acc={row['val_accuracy']:.4f}  "
          f"macro-F1={row['val_macro_f1']:.4f}  AUC={row['val_roc_auc_ovr']:.4f}  "
          f"time={row['train_time_s']}s")
    return row

def run_sklearn(exp_id, name, model, split, preprocessing, settings, notes=""):
    d = SPLITS[split]
    start = time.time()
    model.fit(d["Xtr"], d["ytr"])
    elapsed = time.time() - start
    log_row(exp_id, name, preprocessing, settings, d["desc"],
            d["yva"], model.predict(d["Xva"]), model.predict_proba(d["Xva"]), elapsed, notes)
    return model

def show_log():
    cols = ["exp_id", "model", "val_accuracy", "val_macro_f1", "val_macro_precision",
            "val_macro_recall", "val_roc_auc_ovr", "train_time_s"]
    return pd.DataFrame(experiment_log)[cols].round(4)

_synced = set()
def sync_tensorboard(root=f"{LOG_DIR}/sklearn"):
    """Write each scikit-learn experiment to TensorBoard (deep learning runs log themselves)."""
    keys = ["val_accuracy", "val_macro_f1", "val_macro_precision",
            "val_macro_recall", "val_roc_auc_ovr", "train_time_s"]
    for row in experiment_log:
        if row["exp_id"] in _synced or row["exp_id"].startswith("D"):
            continue
        writer = tf.summary.create_file_writer(f"{root}/{row['exp_id']}")
        with writer.as_default():
            for k in keys:
                tf.summary.scalar(k, row[k], step=0)
            tf.summary.text("settings",
                            f"{row['model']} | {row['preprocessing']} | {row['settings']} | {row['split']}",
                            step=0)
        writer.flush()
        _synced.add(row["exp_id"])

## 6. Traditional ML experiments (E1 to E9)
### E1 to E3: baselines on the random split
Logistic Regression (with log transform and scaling), Random Forest (optional), and Histogram Gradient Boosting. All use class weights because of the imbalance. Expected output: one line per model; scores are very high because the random split leaks neighbouring pixels.

In [ ]:
lr = Pipeline([
    ("signed_log", FunctionTransformer(signed_log1p)),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=300, class_weight="balanced", random_state=SEED)),
])
hgb = HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED)

run_sklearn("E1", "Logistic Regression", clone(lr), "random",
            "signed-log1p + StandardScaler", "max_iter=300, class_weight=balanced", "Linear baseline")

if RUN_RANDOM_FOREST:
    rf = RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample",
                                n_jobs=-1, random_state=SEED)
    run_sklearn("E2", "Random Forest", rf, "random", "none (raw features)",
                "n_estimators=200, class_weight=balanced_subsample", "Tree ensemble baseline")

run_sklearn("E3", "Hist Gradient Boosting", clone(hgb), "random",
            "none (raw features)", "defaults, class_weight=balanced", "Boosting baseline")

### E4 and E5: same models on the block split
This is the honest test of generalisation to new areas. Expected output: scores drop a little (macro-F1 about 0.97), which shows how much the random split overstated performance.

In [ ]:
run_sklearn("E4", "Logistic Regression (block)", clone(lr), "block",
            "signed-log1p + StandardScaler", "same as E1", "Stricter split")
run_sklearn("E5", "Hist Gradient Boosting (block)", clone(hgb), "block",
            "none (raw features)", "same as E3", "Stricter split")

### E6 and E7: date-difference features
Add (14 July minus 5 July) for every feature, to capture crop growth between dates. Expected output: almost no change from E4 and E5.

In [ ]:
hgb_diff = Pipeline([("diffs", FunctionTransformer(add_date_diffs)),
                     ("clf", HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED))])
lr_diff = Pipeline([("diffs", FunctionTransformer(add_date_diffs)),
                    ("signed_log", FunctionTransformer(signed_log1p)),
                    ("scaler", StandardScaler()),
                    ("clf", LogisticRegression(max_iter=300, class_weight="balanced", random_state=SEED))])

run_sklearn("E6", "HGB + date diffs (block)", hgb_diff, "block",
            "date-difference features (+87)", "same as E5", "Feature engineering")
run_sklearn("E7", "Logistic Regression + date diffs (block)", lr_diff, "block",
            "date diffs + signed-log1p + StandardScaler", "same as E4", "Feature engineering")

### E8 and E9: remove redundant features
Drop one feature from every pair with correlation above 0.95. The pairs are found on the training block only. Expected output: about 102 features kept; boosting stays the same and is faster, logistic regression gets a bit worse.

In [ ]:
corr_tr = Xb_train.corr().abs()
upper_tr = corr_tr.where(np.triu(np.ones(corr_tr.shape), k=1).astype(bool))
drop_cols = [c for c in upper_tr.columns if (upper_tr[c] > 0.95).any()]
keep_cols = [c for c in feature_cols if c not in drop_cols]
print(f"Dropped {len(drop_cols)} features, kept {len(keep_cols)}")

SPLITS["block_sel"] = dict(Xtr=Xb_train[keep_cols], ytr=yb_train,
                           Xva=Xb_val[keep_cols], yva=yb_val,
                           desc="within-class blocks 70/15/15, 1% buffers")

run_sklearn("E8", "HGB + correlation filter (block)", clone(hgb), "block_sel",
            f"correlation filter |r|>0.95 ({len(keep_cols)} features)", "same as E5", "Feature selection")
run_sklearn("E9", "Logistic Regression + correlation filter (block)", clone(lr), "block_sel",
            f"correlation filter + signed-log1p + StandardScaler ({len(keep_cols)} features)",
            "same as E4", "Feature selection")

## 7. Tune the boosting model (T1 to T8)
Random search over learning rate, tree size, L2 penalty, and minimum samples per leaf, scored on the block-split validation set. Random k-fold is avoided because it would bring the leakage back. Expected output: one line per setting; scores differ by only about 0.004.

In [ ]:
space = {
    "learning_rate": [0.05, 0.1, 0.2],
    "max_leaf_nodes": [31, 63, 127],
    "l2_regularization": [0.0, 1.0, 10.0],
    "min_samples_leaf": [20, 50, 200],
}
configs = list(ParameterSampler(space, n_iter=N_TUNING_RUNS, random_state=SEED))

tuned = {}
for i, params in enumerate(configs, start=1):
    model = HistGradientBoostingClassifier(**params, max_iter=200,
                                           class_weight="balanced", random_state=SEED)
    tuned[f"T{i}"] = run_sklearn(f"T{i}", f"HGB tuning #{i} (block)", model, "block_sel",
                                 f"correlation filter ({len(keep_cols)} features)",
                                 f"{params}, max_iter=200, class_weight=balanced", "Random search")

tune_rows = [r for r in experiment_log if r["exp_id"].startswith("T")]
best_id = max(tune_rows, key=lambda r: r["val_macro_f1"])["exp_id"]
best_hgb = tuned[best_id]
print("\nBest tuned model:", best_id, "|", [r["settings"] for r in tune_rows if r["exp_id"] == best_id][0])

## 8. Deep learning (D1 to D5)
### Data pipeline
Inputs get the same log transform and scaling, fitted on the training block only. Data flows through `tf.data` (shuffle, batch, prefetch). Class weights counter the imbalance. Expected output: the class weights (largest for classes 2 and 7).

In [ ]:
dl_prep = Pipeline([("signed_log", FunctionTransformer(signed_log1p)),
                    ("scaler", StandardScaler())])
Xtr = dl_prep.fit_transform(Xb_train).astype("float32")
Xva = dl_prep.transform(Xb_val).astype("float32")
ytr = (yb_train.to_numpy() - 1).astype("int32")     # Keras wants labels 0-6
yva = (yb_val.to_numpy() - 1).astype("int32")

classes = np.arange(7)
class_weights = {int(c): float(w) for c, w in
                 zip(classes, compute_class_weight("balanced", classes=classes, y=ytr))}
print({k + 1: round(v, 2) for k, v in class_weights.items()})

BATCH = 512
def make_ds(X, y, training):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:   # buffer = whole set: the block split is stored class by class
        ds = ds.shuffle(buffer_size=len(X), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

### Models and training helpers
Two builders: a Sequential MLP, and a Functional model with separate radar and optical branches. Training stops early on either validation loss or validation macro-F1, and the best epoch's weights are restored. Expected output: none.

In [ ]:
def build_mlp(units=(256, 128, 64), dropout=0.3, lr=1e-3, l2=0.0, name="mlp"):
    reg = keras.regularizers.l2(l2) if l2 > 0 else None
    model = keras.Sequential(name=name)
    model.add(layers.Input(shape=(Xtr.shape[1],)))
    for u in units:
        model.add(layers.Dense(u, activation="relu", kernel_regularizer=reg))
        model.add(layers.Dropout(dropout))
    model.add(layers.Dense(7, activation="softmax"))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

N_RADAR, N_OPTICAL = 98, 76   # f1-f98 radar, f99-f174 optical

def build_two_branch(radar_units=(128, 64), optical_units=(128, 64), head_units=(64,),
                     dropout=0.3, lr=1e-3, name="two_branch"):
    """Functional API: one branch per sensor, joined before the output layer."""
    inp = keras.Input(shape=(N_RADAR + N_OPTICAL,), name="features")
    x = layers.Reshape((N_RADAR + N_OPTICAL, 1))(inp)
    radar = layers.Flatten()(layers.Cropping1D((0, N_OPTICAL))(x))
    optical = layers.Flatten()(layers.Cropping1D((N_RADAR, 0))(x))
    for u in radar_units:
        radar = layers.Dropout(dropout)(layers.Dense(u, activation="relu")(radar))
    for u in optical_units:
        optical = layers.Dropout(dropout)(layers.Dense(u, activation="relu")(optical))
    h = layers.Concatenate()([radar, optical])
    for u in head_units:
        h = layers.Dropout(dropout)(layers.Dense(u, activation="relu")(h))
    out = layers.Dense(7, activation="softmax")(h)
    model = keras.Model(inp, out, name=name)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

class MacroF1Stopping(keras.callbacks.Callback):
    """Stop when validation macro-F1 stops improving; restore the best epoch's weights."""
    def __init__(self, X_val, y_val, patience=6):
        super().__init__()
        self.X_val, self.y_val, self.patience = X_val, y_val, patience
    def on_train_begin(self, logs=None):
        self.best, self.wait, self.best_weights, self.best_epoch = -1.0, 0, None, 0
    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(self.X_val, batch_size=2048, verbose=0).argmax(axis=1)
        f1 = f1_score(self.y_val, pred, average="macro")
        print(f"   val_macro_f1={f1:.4f}")
        if f1 > self.best:
            self.best, self.wait, self.best_epoch = f1, 0, epoch + 1
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)
        print(f"   restored weights from epoch {self.best_epoch} (val macro-F1 {self.best:.4f})")

def get_class_weights(scheme):
    if scheme == "balanced":
        return class_weights
    if scheme == "sqrt":
        return {k: float(v ** 0.5) for k, v in class_weights.items()}
    return None

def run_dl_experiment(exp_id, name, model, settings, notes="", epochs=40,
                      patience=5, cw="balanced", stop_on="val_loss"):
    train_ds, val_ds = make_ds(Xtr, ytr, True), make_ds(Xva, yva, False)
    callbacks = [keras.callbacks.TensorBoard(log_dir=f"{LOG_DIR}/dl/{exp_id}")]
    if stop_on == "macro_f1":
        callbacks.insert(0, MacroF1Stopping(Xva, yva, patience=patience))
    else:
        callbacks.append(keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True))
    start = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                        class_weight=get_class_weights(cw), callbacks=callbacks, verbose=2)
    elapsed = time.time() - start
    proba = model.predict(val_ds, verbose=0)
    log_row(exp_id, name, "signed-log1p + StandardScaler (174 features)",
            f"{settings}, class_weight={cw}, stop_on={stop_on}, batch={BATCH}, "
            f"params={model.count_params()}, epochs_run={len(history.history['loss'])}",
            SPLITS["block"]["desc"], yva, proba.argmax(axis=1), proba, elapsed, notes)
    return model, history

def plot_history(history, title):
    h = history.history
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(h["loss"], label="train"); axes[0].plot(h["val_loss"], label="validation")
    axes[0].set_title(f"{title}: loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(h["accuracy"], label="train"); axes[1].plot(h["val_accuracy"], label="validation")
    axes[1].set_title(f"{title}: accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
    plt.tight_layout(); plt.show()

### D1: baseline MLP
Three hidden layers (256, 128, 64), dropout 0.3, Adam. Expected output: training loss keeps falling while validation loss rises after a few epochs (overfitting).

In [ ]:
tf.keras.utils.set_random_seed(SEED)
mlp1 = build_mlp(units=(256, 128, 64), dropout=0.3, lr=1e-3, name="mlp_d1")
mlp1, hist1 = run_dl_experiment("D1", "MLP 256-128-64", mlp1,
    "units=(256,128,64), dropout=0.3, Adam lr=1e-3, patience=5", "Deep learning baseline")
plot_history(hist1, "D1")

### D2: softer class weights
Same as D1 but with square-root class weights. Expected output: no real change (the weights were not the main problem).

In [ ]:
tf.keras.utils.set_random_seed(SEED)
mlp2 = build_mlp(units=(256, 128, 64), dropout=0.3, lr=1e-3, name="mlp_d2")
mlp2, hist2 = run_dl_experiment("D2", "MLP 256-128-64, sqrt class weights", mlp2,
    "units=(256,128,64), dropout=0.3, Adam lr=1e-3, patience=5", "Class weights vs D1", cw="sqrt")
plot_history(hist2, "D2")

### D3: more regularisation
Dropout 0.5 plus L2 weight decay. Expected output: a smaller train/validation gap but a lower macro-F1.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
mlp3 = build_mlp(units=(256, 128, 64), dropout=0.5, lr=1e-3, l2=1e-4, name="mlp_d3")
mlp3, hist3 = run_dl_experiment("D3", "MLP 256-128-64, dropout 0.5 + L2 1e-4", mlp3,
    "units=(256,128,64), dropout=0.5, l2=1e-4, Adam lr=1e-3, patience=5", "Regularisation vs D1")
plot_history(hist3, "D3")

### D4: stop on macro-F1
Same as D1, but training stops on validation macro-F1 instead of validation loss. Expected output: a better macro-F1 than D1. Note that macro-F1 jumps around between epochs, so the best epoch is a slightly optimistic pick.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
mlp4 = build_mlp(units=(256, 128, 64), dropout=0.3, lr=1e-3, name="mlp_d4")
mlp4, hist4 = run_dl_experiment("D4", "MLP 256-128-64, macro-F1 stopping", mlp4,
    "units=(256,128,64), dropout=0.3, Adam lr=1e-3, patience=6", "Stopping rule vs D1",
    patience=6, stop_on="macro_f1")
plot_history(hist4, "D4")

### D5: two-branch Functional model
One branch for radar features and one for optical features, joined before the output. Same training setup as D4. Expected output: similar to D4, with no clear gain.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
mlp5 = build_two_branch(dropout=0.3, lr=1e-3, name="two_branch_d5")
mlp5, hist5 = run_dl_experiment("D5", "Two-branch Functional MLP", mlp5,
    "radar 128-64, optical 128-64, fusion 64, dropout=0.3, Adam lr=1e-3, patience=6",
    "Architecture vs D4", patience=6, stop_on="macro_f1")
plot_history(hist5, "D5")

## 9. Experiment table
All runs in one table, saved to `experiment_log.csv` (full settings included) and to TensorBoard. Expected output: one row per experiment.

In [ ]:
sync_tensorboard()
pd.DataFrame(experiment_log).to_csv("experiment_log.csv", index=False)
pd.set_option("display.width", 200)
show_log()

## 10. Final test-set evaluation
The test set was not used for any decision. Three models are scored once: the best tuned boosting model, D4, and D5. Expected output: a table, per-class reports, and confusion matrices. Accuracy stays high, but macro-F1 drops because class 7 loses precision.

In [ ]:
Xte = dl_prep.transform(Xb_test).astype("float32")
yte = yb_test.to_numpy()

def dl_predict(model):
    proba = model.predict(Xte, batch_size=2048, verbose=0)
    return proba.argmax(axis=1) + 1, proba     # back to labels 1-7

Xb_test_sel = Xb_test[keep_cols]
candidates = {
    f"Tuned boosting ({best_id})": (best_hgb.predict(Xb_test_sel), best_hgb.predict_proba(Xb_test_sel)),
    "D4 MLP (macro-F1 stopping)": dl_predict(mlp4),
    "D5 two-branch Functional": dl_predict(mlp5),
}

rows = []
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, (pred, proba)) in zip(axes, candidates.items()):
    rows.append({
        "model": name,
        "test_accuracy": accuracy_score(yte, pred),
        "test_macro_f1": f1_score(yte, pred, average="macro"),
        "test_macro_precision": precision_score(yte, pred, average="macro", zero_division=0),
        "test_macro_recall": recall_score(yte, pred, average="macro", zero_division=0),
        "test_roc_auc_ovr": roc_auc_score(yte, proba, multi_class="ovr", average="macro"),
    })
    print("===", name, "===")
    print(classification_report(yte, pred, digits=4))
    ConfusionMatrixDisplay.from_predictions(yte, pred, normalize="true", values_format=".3f",
                                            cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()

test_table = pd.DataFrame(rows).round(4)
test_table.to_csv("test_results.csv", index=False)
test_table

## 11. Error analysis: why class 7 precision is low
Class 7 has almost perfect recall but low precision on the test block. This cell counts the false positives, shows which true classes they come from, and plots the class 7 precision-recall curve. Expected output: a table and a plot. Most false positives come from corn (class 1).

In [ ]:
true7 = (yte == 7).astype(int)
fp_rows = []
fig, ax = plt.subplots(figsize=(6, 5))
for name, (pred, proba) in candidates.items():
    ap = average_precision_score(true7, proba[:, 6])
    fp_mask = (pred == 7) & (yte != 7)
    fp_rows.append({"model": name, "class7_AP": round(ap, 4),
                    "predicted_as_7": int((pred == 7).sum()), "true_7": int(true7.sum()),
                    "false_positives": int(fp_mask.sum()),
                    "FP_by_true_class": pd.Series(yte[fp_mask]).value_counts().sort_index().to_dict()})
    prec, rec, _ = precision_recall_curve(true7, proba[:, 6])
    ax.plot(rec, prec, label=f"{name} (AP={ap:.3f})")
ax.set_xlabel("Recall (class 7)"); ax.set_ylabel("Precision (class 7)")
ax.set_title("Class 7 precision-recall, test set"); ax.legend(loc="lower left")
plt.tight_layout(); plt.show()
pd.DataFrame(fp_rows)

## 12. Save the best models
The boosting model is saved with joblib (plus its feature list). The MLP is saved as `.keras` and `.h5` (plus its scaler). A quick reload check confirms the saved files give the same predictions. To use the MLP on new data: apply `sign(x)*log1p(|x|)`, then the saved scaler, then the model. Expected output: the file list and two "True" checks.

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(best_hgb, f"{MODEL_DIR}/best_traditional_hgb.joblib")
with open(f"{MODEL_DIR}/best_traditional_hgb_features.json", "w") as f:
    json.dump(keep_cols, f)
mlp4.save(f"{MODEL_DIR}/best_dl_mlp_D4.keras")
mlp4.save(f"{MODEL_DIR}/best_dl_mlp_D4.h5")      # older format; Keras may print a warning
mlp5.save(f"{MODEL_DIR}/D5_two_branch.keras")
joblib.dump(dl_prep.named_steps["scaler"], f"{MODEL_DIR}/dl_standard_scaler.joblib")
print(sorted(os.listdir(MODEL_DIR)))

# reload check
hgb_loaded = joblib.load(f"{MODEL_DIR}/best_traditional_hgb.joblib")
mlp_loaded = keras.models.load_model(f"{MODEL_DIR}/best_dl_mlp_D4.keras")
print("Boosting reload matches:", bool((hgb_loaded.predict(Xb_test_sel) == candidates[list(candidates)[0]][0]).all()))
print("MLP reload matches:", bool((mlp_loaded.predict(Xte, batch_size=2048, verbose=0).argmax(axis=1) + 1 == candidates["D4 MLP (macro-F1 stopping)"][0]).all()))

### Copy to Google Drive
Colab wipes its disk when the session ends, so copy the models, logs, tables, and the dataset zip to Drive. Expected output: the Drive folder contents (skipped outside Colab).

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/crop_mapping_project"
    os.makedirs(dest, exist_ok=True)
    shutil.copytree(MODEL_DIR, f"{dest}/models", dirs_exist_ok=True)
    shutil.copytree(LOG_DIR, f"{dest}/logs", dirs_exist_ok=True)
    for f in [ZIP_PATH, "experiment_log.csv", "test_results.csv"]:
        shutil.copy(f, dest)
    print(sorted(os.listdir(dest)))
except ImportError:
    print("Not running in Colab: skipping the Drive copy.")

## 13. TensorBoard
Opens the logged runs (scikit-learn results and the deep learning curves). Expected output: an embedded TensorBoard view.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs